<a href="https://colab.research.google.com/github/mrayhanz/244107020027-Machine-Learning/blob/main/Minggu%206/MRayhanZamzami_JS06_ANN_P4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IDENTITAS DIRI

Nama : Muhammad Rayhan Zamzami

NIM : 244107020027

Kelas : TI-3G

Absen : 19

# PRAKTIKUM 4 - Perbandingan Annoy, FAISS, dan HNSW


In [1]:
# Instalasi pustaka ANN
!pip install annoy faiss-cpu hnswlib

### 1. Implementasi Perbandingan pada 1 Juta Data 5D

In [2]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
np.random.seed(42)
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

=== Annoy ===


Build time: 4.993584871292114 detik
Query time: 0.03775358200073242 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.008476018905639648 detik
Query time: 0.005407810211181641 detik
Neighbors: [901095 495168 561017 163986 185734] ...

=== HNSW (hnswlib) ===


Build time: 2.945000410079956 detik
Query time: 0.021750450134277344 detik
Neighbors: [901095 495168 561017 163986 185734] ...


### 2. Eksperimen Variasi Metrik, Jumlah Data, dan Dimensi


In [3]:
import pandas as pd

def run_annoy(X_data, q, k, metric):
    metric_annoy = 'euclidean' if metric == 'euclidean' else 'angular'
    idx = AnnoyIndex(X_data.shape[1], metric_annoy)
    t0 = time.time()
    for i, v in enumerate(X_data):
        idx.add_item(i, v)
    idx.build(10)
    build = time.time() - t0
    t0 = time.time()
    res = idx.get_nns_by_vector(q[0], k)
    query = time.time() - t0
    return build, query, res

def run_faiss(X_data, q, k, metric):
    if metric == 'euclidean':
        idx = faiss.IndexFlatL2(X_data.shape[1])
        metric_faiss = faiss.METRIC_L2
    else:
        faiss.normalize_L2(X_data)
        faiss.normalize_L2(q)
        idx = faiss.IndexFlatIP(X_data.shape[1])
        metric_faiss = faiss.METRIC_INNER_PRODUCT
    t0 = time.time()
    idx.add(X_data)
    build = time.time() - t0
    t0 = time.time()
    D, I = idx.search(q, k)
    query = time.time() - t0
    return build, query, I[0].tolist()

def run_hnsw(X_data, q, k, metric):
    space = 'l2' if metric == 'euclidean' else 'cosine'
    idx = hnswlib.Index(space=space, dim=X_data.shape[1])
    t0 = time.time()
    idx.init_index(max_elements=X_data.shape[0], ef_construction=200, M=16)
    idx.add_items(X_data)
    build = time.time() - t0
    idx.set_ef(50)
    t0 = time.time()
    labels, _ = idx.knn_query(q, k=k)
    query = time.time() - t0
    return build, query, labels[0].tolist()

def benchmark(n_samples, dim, metric, k=10):
    np.random.seed(42)
    X_data = np.random.random((n_samples, dim)).astype(np.float32)
    q = np.random.random((1, dim)).astype(np.float32)
    row = {"Data": f"{n_samples:,}", "Dimensi": f"{dim}D", "Metric": metric}
    for name, fn in [("Annoy", run_annoy), ("FAISS", run_faiss), ("HNSW", run_hnsw)]:
        build, query_time, res = fn(X_data.copy(), q.copy(), k, metric)
        row[f"{name} Build (s)"] = round(build, 4)
        row[f"{name} Query (ms)"] = round(query_time * 1000, 4)
    return row

configs = [
    (1000, 2, 'euclidean'), (1000, 2, 'cosine'),
    (1000, 5, 'euclidean'), (1000, 5, 'cosine'),
    (100000, 5, 'euclidean'), (100000, 5, 'cosine'),
    (1000000, 5, 'euclidean'), (1000000, 5, 'cosine'),
]

rows = [benchmark(n, d, m) for n, d, m in configs]
df_p4 = pd.DataFrame(rows)
display(df_p4)

,Data,Dimensi,Metric,Annoy Build (s),Annoy Query (ms),FAISS Build (s),FAISS Query (ms),HNSW Build (s),HNSW Query (ms)
0,"1,000",2D,euclidean,0.0019,17.8096,0.0000,0.1500,0.0017,18.3249
1,"1,000",2D,cosine,0.0019,31.7702,0.0000,0.0346,0.0006,32.3904
2,"1,000",5D,euclidean,0.0025,17.2017,0.0000,0.0608,0.0013,18.2431
3,"1,000",5D,cosine,0.0020,22.3470,0.0000,0.0367,0.0008,32.2945
4,"100,000",5D,euclidean,0.2840,18.5490,0.0006,2.4464,0.1854,18.9133
5,"100,000",5D,cosine,0.1701,94.2397,0.0012,0.4132,0.0021,52.2094
6,"1,000,000",5D,euclidean,4.7442,19.4945,0.0090,3.9284,2.9471,22.2785
7,"1,000,000",5D,cosine,1.7838,52.9609,0.0071,0.9654,0.0101,68.7981
